### Importaciones Genericas

In [85]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from pandas.api.types import CategoricalDtype
%matplotlib inline
sns.set(color_codes=True)

### Cargar Datos (Dataset)

In [86]:
df = pd.read_csv("../data/02_intermediate/weatherAUS_Cleaned.csv")
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Cloud3pm,Temp9am,Temp3pm,RainToday,RISK_MM,RainTomorrow,Year,Month,Month_Name,Location_encoded
0,2008-12-01,Albury,13.4,22.9,0.6,6.447059,7.658013,W,44.0,W,...,NaN,16.9,21.8,0,0.0,0,2008,12,December,2
1,2008-12-02,Albury,7.4,25.1,0.0,6.447059,7.658013,WNW,44.0,NNW,...,NaN,17.2,24.3,0,0.0,0,2008,12,December,2
2,2008-12-03,Albury,12.9,25.7,0.0,6.447059,7.658013,WSW,46.0,W,...,2.0,21.0,23.2,0,0.0,0,2008,12,December,2
3,2008-12-04,Albury,9.2,28.0,0.0,6.447059,7.658013,NE,24.0,SE,...,5.0,18.1,26.5,0,1.0,0,2008,12,December,2
4,2008-12-05,Albury,17.5,32.3,1.0,6.447059,7.658013,W,41.0,ENE,...,8.0,17.8,29.7,0,0.2,0,2008,12,December,2


In [87]:
print(df.isnull().sum())

Date                    0
Location                0
MinTemp                 0
MaxTemp                 0
Rainfall                0
Evaporation             0
Sunshine                0
WindGustDir          9330
WindGustSpeed           0
WindDir9am          10013
WindDir3pm           3778
WindSpeed9am            0
WindSpeed3pm            0
Humidity9am             0
Humidity3pm             0
Pressure9am             0
Pressure3pm             0
Cloud9am                0
Cloud3pm                2
Temp9am                 0
Temp3pm                 0
RainToday               0
RISK_MM                 0
RainTomorrow            0
Year                    0
Month                   0
Month_Name              0
Location_encoded        0
dtype: int64


## Limpieza de datos

In [88]:
df_Cleaned= df.copy()

### Imputacion por valores faltantes de WindGustDir

In [89]:
print(df['WindGustDir'].value_counts())
print("Valores nulos:", df['WindGustDir'].isnull().sum())

WindGustDir
W      9780
SE     9309
E      9071
N      9033
SSE    8993
S      8949
WSW    8901
SW     8797
SSW    8610
WNW    8066
NW     8003
ENE    7992
ESE    7305
NE     7060
NNW    6561
NNE    6433
Name: count, dtype: int64
Valores nulos: 9330


In [90]:
df_Cleaned= df.copy()

df_Cleaned['WindGustDir'] = df_Cleaned['WindGustDir'].fillna(df_Cleaned['WindDir9am'])

df_Cleaned['WindGustDir'] = df_Cleaned['WindGustDir'].fillna(df_Cleaned['WindDir3pm'])

moda = df_Cleaned['WindGustDir'].mode()[0]
df_Cleaned['WindGustDir'] = df_Cleaned['WindGustDir'].fillna(moda)

# Verificación
print("Valores nulos en WindGustDir después de imputación:", df_Cleaned['WindGustDir'].isnull().sum())

Valores nulos en WindGustDir después de imputación: 0


### Renombre de Variables

In [91]:
# Renombrar la variables a un termino más adecuado a su uso y a lo que representan
df_Cleaned = df_Cleaned.rename(columns={
    'Location': 'Location_name',
    'WindGustDir': 'WindDir_avg',
    'WindGustSpeed': 'WindSpeed_max',
})

# Verificar
print(df_Cleaned.columns)

Index(['Date', 'Location_name', 'MinTemp', 'MaxTemp', 'Rainfall',
       'Evaporation', 'Sunshine', 'WindDir_avg', 'WindSpeed_max', 'WindDir9am',
       'WindDir3pm', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am',
       'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm',
       'Temp9am', 'Temp3pm', 'RainToday', 'RISK_MM', 'RainTomorrow', 'Year',
       'Month', 'Month_Name', 'Location_encoded'],
      dtype='object')


### Reducción de variables

#### Crear nuevas variables promedio


In [92]:
# Calcular promedios (ignorando nulos si los hay)
df_Cleaned['WindSpeed_avg'] = df_Cleaned[['WindSpeed9am', 'WindSpeed3pm']].mean(axis=1)
df_Cleaned['Humidity_avg'] = df_Cleaned[['Humidity9am', 'Humidity3pm']].mean(axis=1)
df_Cleaned['Pressure_avg'] = df_Cleaned[['Pressure9am', 'Pressure3pm']].mean(axis=1)
df_Cleaned['Cloud_avg'] = df_Cleaned[['Cloud9am', 'Cloud3pm']].mean(axis=1)
df_Cleaned['Temp_avg'] = df_Cleaned[['Temp9am', 'Temp3pm']].mean(axis=1)


# Verificar las nuevas columnas
print(df_Cleaned[['WindSpeed_avg', 'Humidity_avg', 'Pressure_avg', 'Cloud_avg', 'Temp_avg']].head())

   WindSpeed_avg  Humidity_avg  Pressure_avg  Cloud_avg  Temp_avg
0           22.0          46.5       1007.40      8.000     19.35
1           13.0          34.5       1009.20      7.750     20.75
2           22.5          34.0       1008.15      4.750     22.10
3           10.0          30.5       1015.20      6.125     22.30
4           13.5          57.5       1008.40      7.500     23.75


### Eliminación de columas no necesarias

Eliminar
- Date
- WindDir9am
- WindDir3pm
- WindSpeed9am
- WindSpeed3pm
- Humidity9am
- Humidity3pm
- Pressure9am
- Pressure3pm
- Cloud9am
- Cloud3pm
- Temp9am
- Temp3pm

In [93]:
df_Cleaned.drop(
    columns=[
        'WindDir9am', 'WindDir3pm',
        'WindSpeed9am','WindSpeed3pm',
        'Humidity9am', 'Humidity3pm',
        'Pressure9am', 'Pressure3pm',
        'Cloud9am', 'Cloud3pm',
        'Temp9am', 'Temp3pm',
    ],
    inplace=True
)

print("\nColumnas finales:", df_Cleaned.columns.tolist())


Columnas finales: ['Date', 'Location_name', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindDir_avg', 'WindSpeed_max', 'RainToday', 'RISK_MM', 'RainTomorrow', 'Year', 'Month', 'Month_Name', 'Location_encoded', 'WindSpeed_avg', 'Humidity_avg', 'Pressure_avg', 'Cloud_avg', 'Temp_avg']


### Reordenar Columnas

In [97]:
# Definir el nuevo orden de columnas
nuevo_orden = [
    # 1. Datos temporales
    'Date', 'Year', 'Month', 'Month_Name',
    # 2. Ubicación
    'Location_encoded','Location_name', 
    # 3. Variables climáticas (temperaturas, humedad, viento, etc.)
    'MinTemp', 'MaxTemp', 'Temp_avg',
    'Rainfall', 'Evaporation', 'Sunshine',
    'WindDir_avg', 'WindSpeed_max', 'WindSpeed_avg',
    'Humidity_avg', 'Pressure_avg', 'Cloud_avg',
    # 4. Variables binarias/días específicos
    'RainToday',
    # 5. Target y riesgo
    'RISK_MM', 'RainTomorrow',
]
# Reordenar el DataFrame
df_Cleaned = df_Cleaned[nuevo_orden]
# Verificar
print("Columnas ordenadas:", df_Cleaned.columns.tolist())

Columnas ordenadas: ['Date', 'Year', 'Month', 'Month_Name', 'Location_encoded', 'Location_name', 'MinTemp', 'MaxTemp', 'Temp_avg', 'Rainfall', 'Evaporation', 'Sunshine', 'WindDir_avg', 'WindSpeed_max', 'WindSpeed_avg', 'Humidity_avg', 'Pressure_avg', 'Cloud_avg', 'RainToday', 'RISK_MM', 'RainTomorrow']


### Post limpieza

In [98]:
print(df_Cleaned.isnull().sum())

Date                0
Year                0
Month               0
Month_Name          0
Location_encoded    0
Location_name       0
MinTemp             0
MaxTemp             0
Temp_avg            0
Rainfall            0
Evaporation         0
Sunshine            0
WindDir_avg         0
WindSpeed_max       0
WindSpeed_avg       0
Humidity_avg        0
Pressure_avg        0
Cloud_avg           0
RainToday           0
RISK_MM             0
RainTomorrow        0
dtype: int64


In [99]:
print(df_Cleaned.dtypes)

Date                 object
Year                  int64
Month                 int64
Month_Name           object
Location_encoded      int64
Location_name        object
MinTemp             float64
MaxTemp             float64
Temp_avg            float64
Rainfall            float64
Evaporation         float64
Sunshine            float64
WindDir_avg          object
WindSpeed_max       float64
WindSpeed_avg       float64
Humidity_avg        float64
Pressure_avg        float64
Cloud_avg           float64
RainToday             int64
RISK_MM             float64
RainTomorrow          int64
dtype: object


In [100]:
df_Cleaned.head()

,Date,Year,Month,Month_Name,Location_encoded,Location_name,MinTemp,MaxTemp,Temp_avg,Rainfall,...,Sunshine,WindDir_avg,WindSpeed_max,WindSpeed_avg,Humidity_avg,Pressure_avg,Cloud_avg,RainToday,RISK_MM,RainTomorrow
0,2008-12-01,2008,12,December,2,Albury,13.4,22.9,19.35,0.6,...,7.658013,W,44.0,22.0,46.5,1007.40,8.000,0,0.0,0
1,2008-12-02,2008,12,December,2,Albury,7.4,25.1,20.75,0.0,...,7.658013,WNW,44.0,13.0,34.5,1009.20,7.750,0,0.0,0
2,2008-12-03,2008,12,December,2,Albury,12.9,25.7,22.10,0.0,...,7.658013,WSW,46.0,22.5,34.0,1008.15,4.750,0,0.0,0
3,2008-12-04,2008,12,December,2,Albury,9.2,28.0,22.30,0.0,...,7.658013,NE,24.0,10.0,30.5,1015.20,6.125,0,1.0,0
4,2008-12-05,2008,12,December,2,Albury,17.5,32.3,23.75,1.0,...,7.658013,W,41.0,13.5,57.5,1008.40,7.500,0,0.2,0
